In [ ]:
library(Seurat)
library(ggplot2)
library(pheatmap)
library(dplyr)
library(RColorBrewer)
library(clustree)
library(reshape2)
library(ggpubr)
getwd()
dir.create("figures_10xPBMC")
dir.create("data_10xPBMC")
dataset_id <- "10xPBMC"
sample_id <- "pbmc8k"
colorPBMC <- "#81B29A"


In [ ]:
# celltype annotation

celltypes <- read.table("TEbenchmarking/data/whitelists/celltype_annotation_sub_min.tsv") # same file passed to snakemake

celltypes$V2 <- gsub(pattern="CD14pos_Monocytes", replacement = "Monocytes_CD14pos", celltypes$V2)
celltypes$V2 <- gsub(pattern="FCGR3Apos_Monocytes", replacement = "Monocytes_FCGR3Apos", celltypes$V2)
celltypes$V2 <- gsub(pattern="Naive_CD4_T", replacement = "T_CD4_Naive", celltypes$V2)
celltypes$V2 <- gsub(pattern="Memory_CD4_T", replacement = "T_CD4_Memory", celltypes$V2)
celltypes$V2 <- gsub(pattern="CD8A", replacement = "T_CD8", celltypes$V2)
celltypes$V2 <- gsub(pattern="Megakaryocytes", replacement = "Platelet", celltypes$V2)

PBMCcelltypeColors <- c("B"="#91bcca",
                "Monocytes_CD14pos"="#3d405b",
                "T_CD8"="#f2cc8f",
                "Dendritic"="#c492a7",
                "Monocytes_FCGR3Apos"="#e78063",
               # "Platelet"="#c4b3ab",
                "T_CD4_Memory"="#81b29a",
                "T_CD4_Naive"="#4d7362", 
                "NK"= "#88535a")

In [ ]:
getwd()

conversionTable <- read.table("annotation/annotation_hg38_conversion_withAge.tsv")
head(conversionTable)

In [ ]:
SoloTE_path <- paste0("/mnt/volume_1p5T/results/SoloTEout/", dataset_id, "/", sample_id, "/", sample_id,"_SoloTE_output/", sample_id,"_legacytes_MATRIX")
STAR_path <- paste0("/mnt/TEresults/snakemake_results/results/STARoutdir/",dataset_id,"/",sample_id,"/best_Solo.out/Gene")

legacyTEmatrix <- Seurat::ReadMtx(mtx = paste0(SoloTE_path, "/matrix.mtx"), 
                              cells = paste0(SoloTE_path, "/barcodes.tsv"), 
                              features = paste0(SoloTE_path, "/features.tsv")) # read matrix

# select TEs
TEs <- grep("SoloTE", rownames(legacyTEmatrix), value = T)
locusTEs <- grep("chr", TEs, value = T)
# subset the matrix keeping only TEs
TEmatrix <- legacyTEmatrix[locusTEs,]
# remove "SoloTE" from the name of the TEs
rownames(TEmatrix) <- gsub("SoloTE\\|", "", rownames(TEmatrix))

rownames(TEmatrix) <- gsub("\\|", "-", rownames(TEmatrix))
rownames(TEmatrix) <- gsub("\\_", "-", rownames(TEmatrix))
rownames(TEmatrix) <- gsub("\\?", "", rownames(TEmatrix))

#table(rownames(TEmatrix) %in% conversionTable$soloteID)
# transform into Stellarscope IDs
#rownames(TEmatrix) <- conversionTable$stellarscopeID[match(rownames(TEmatrix), conversionTable$soloteID)]
nCells <- ncol(TEmatrix)
thrMinCells <- round(nCells * 0.05)

# create Seurat object with shallow filtering of TEs expressed in at least 5% of cells and cells expressing at least 50 TEs 
objTE <- Seurat::CreateSeuratObject(TEmatrix, project = "PBMC",
                                    min.cells = thrMinCells, min.features = 50) 
objTE

In [ ]:
# select genes
genes <- setdiff(rownames(legacyTEmatrix), TEs)
# subset the matrix keeping only genes
Genematrix <- legacyTEmatrix[genes,]
# create Seurat object with shallow filtering of genes expressed in at least 50 cells and cells expressing at least 100 genes 
Gene <- Seurat::CreateSeuratObject(Genematrix, project = "PBMC", 
                          min.cells = thrMinCells, min.features = 100) 

filteredBarcodes <- read.table(paste0(STAR_path, "/filtered/barcodes.tsv"))$V1 # read barcodes seleced by STARsolo

# barcodes of platelets
plateletBarcodes <- celltypes[celltypes$V2=="Platelet","V1"]

# remove low quality cells and platelets
objTE_SoloTE <- objTE[,setdiff(filteredBarcodes, plateletBarcodes)] # keep only filtered barcodes
objGenes_SoloTE <- Gene[,setdiff(filteredBarcodes, plateletBarcodes)] # keep only filtered barcodes

objTE_SoloTE
objGenes_SoloTE

In [ ]:
# QC TEs

options(repr.plot.width=7, repr.plot.height=6)

objTE_SoloTE@meta.data$nCount_TE <- objTE_SoloTE@meta.data$nCount_RNA 
objTE_SoloTE@meta.data$nFeature_TE <- objTE_SoloTE@meta.data$nFeature_RNA 

# Visualize QC metrics as a violin plot
VlnPlot(objTE_SoloTE, features = c("nCount_RNA", "nFeature_RNA"), ncol = 2, 
        cols = colorPBMC, pt.size = 0, alpha = 0.5) 
summary(objTE_SoloTE$nCount_RNA)
summary(objTE_SoloTE$nFeature_RNA)

In [ ]:
# QC genes
options(repr.plot.width=8, repr.plot.height=5)

VlnPlot(objGenes_SoloTE, features = c( "nCount_RNA", "nFeature_RNA" ), ncol = 2, 
        cols = alpha(colorPBMC, 0.5), pt.size = 0.05, alpha = 0.5) 

summary(objGenes_SoloTE$nFeature_RNA)
summary(objGenes_SoloTE$nCount_RNA)

# Genes

In [ ]:
objGenes_SoloTE <- NormalizeData(objGenes_SoloTE, normalization.method = "LogNormalize", scale.factor = 10000)

objGenes_SoloTE <- FindVariableFeatures(objGenes_SoloTE, selection.method = "vst", nfeatures = 4000)

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objGenes_SoloTE), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objGenes_SoloTE)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:
gc()
all.genes <- rownames(objGenes_SoloTE)
objGenes_SoloTE <- ScaleData(objGenes_SoloTE) # on hvgs

objGenes_SoloTE <- RunPCA(objGenes_SoloTE, features = VariableFeatures(object = objGenes_SoloTE))


DimPlot(objGenes_SoloTE, reduction = "pca") + NoLegend()

ElbowPlot(objGenes_SoloTE)


In [ ]:
objGenes_SoloTE <- FindNeighbors(objGenes_SoloTE, dims = 1:10, k.param = 20)
objGenes_SoloTE <- FindClusters(objGenes_SoloTE, resolution = 1.6)
objGenes_SoloTE <- RunUMAP(objGenes_SoloTE, dims = 1:10)
DimPlot(objGenes_SoloTE, reduction = "umap")

In [ ]:
objGenes_SoloTE$celltype <- celltypes$V2[match(Cells(objGenes_SoloTE), celltypes$V1)]
table(objGenes_SoloTE$celltype)

In [ ]:
options(repr.plot.width=10, repr.plot.height=7)

DimPlot(objGenes_SoloTE, reduction = "umap", group.by = "celltype",
        cols=PBMCcelltypeColors,
        shuffle=T, pt.size = 1) + 
  theme_pubr() +
  theme(text=element_text(size=20)) 
ggsave(paste0("figures_",dataset_id,"/umap_genes_celltypes_SoloTE_noplatelet.pdf"), device = "pdf",width=10, height=7)

In [ ]:
saveRDS(objGenes_SoloTE, file=paste0("data_", dataset_id, "/objGenes_SoloTE_noPlatelet.RDS"))


# TEs 

In [ ]:

### Normalize

objTE_SoloTE <- NormalizeData(objTE_SoloTE, normalization.method = "LogNormalize", scale.factor = 10000)

objTE_SoloTE <- FindVariableFeatures(objTE_SoloTE, selection.method = "vst", nfeatures = 4000)

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objTE_SoloTE), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objTE_SoloTE)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:

gc()
all.genes <- rownames(objTE_SoloTE)
objTE_SoloTE <- ScaleData(objTE_SoloTE) # on hvgs

grep("MERVL", all.genes, value = T)[1:50]


In [ ]:

objTE_SoloTE <- RunPCA(objTE_SoloTE, features = VariableFeatures(object = objTE_SoloTE))


DimPlot(objTE_SoloTE, reduction = "pca") + NoLegend()

ElbowPlot(objTE_SoloTE)

In [ ]:
objTE_SoloTE <- FindNeighbors(objTE_SoloTE, dims = 1:12, k.param = 20)
objTE_SoloTE <- FindClusters(objTE_SoloTE, resolution = 1)
objTE_SoloTE <- RunUMAP(objTE_SoloTE, dims = 1:12)
DimPlot(objTE_SoloTE, reduction = "umap") +   
    theme_void() +
    theme(text=element_text(size=20)) 

# Checkpoint

In [ ]:
#saveRDS(objTE_SoloTE, paste0("data_", dataset_id, "/soloTE_", dataset_id, "_seuratObj_noPlatelet.RDS"))
#objTE_SoloTE <- readRDS(paste0("data_", dataset_id, "/soloTE_", dataset_id, "_seuratObj.RDS"))

In [ ]:
objGenes_SoloTE <- readRDS(paste0("data_", sample_id, "/objGenes_SoloTE_noPlatelet.RDS"))

In [ ]:
#save.image("workspaces/10xPBMC_SoloTE_260119_noplatelet.Rdata")

In [ ]:
load("workspaces/10xPBMC_SoloTE_260119_noplatelet.Rdata")

# Gene - TE comparison

In [ ]:
options(repr.plot.width=9.5, repr.plot.height=7)
library(ggpubr)
library(rcartocolor)

DimPlot(objTE_SoloTE, reduction = "umap", 
        shuffle=T, pt.size = 1) + 
        ggtitle("TE-derived clusters") +
  scale_color_carto_d(palette="Pastel") +
  theme_pubr() +
  theme(text=element_text(size=20), plot.title = element_text(hjust=0.5)) 
ggsave(paste0("figures_",dataset_id,"/umap_TEs_locus_clusters_SoloTE_Pastel_noplatelet.pdf"), device = "pdf", width=9.5, height=7)

In [ ]:
options(repr.plot.width=9.5, repr.plot.height=7)
library(ggpubr)
library(rcartocolor)

DimPlot(objTE_SoloTE, reduction = "umap", 
        shuffle=T, pt.size = 1) + 
        ggtitle("TE-derived clusters") +
  scale_color_carto_d(palette="Pastel") +
  theme_pubr() +
  theme(text=element_text(size=20), plot.title = element_text(hjust=0.5)) 
ggsave(paste0("figures_",dataset_id,"/umap_TEs_locus_clusters_SoloTE_noplatlet.pdf"), device = "pdf", width=9.5, height=7)

DimPlot(objTE_SoloTE, reduction = "umap", 
        shuffle=T, pt.size = 1) + 
        ggtitle("TE-derived clusters") +
  scale_color_brewer(palette="Spectral") +
  theme_pubr() +
  theme(text=element_text(size=20), plot.title = element_text(hjust=0.5)) 

In [ ]:
options(repr.plot.width=9.5, repr.plot.height=7)
library(ggpubr)
library(scico)
DimPlot(objTE_SoloTE, reduction = "umap", 
        shuffle=T, pt.size = 1) + 
        ggtitle("TE-derived clusters") +
  scale_color_manual(values= scico(length(unique(objTE_SoloTE$seurat_clusters)), palette = 'managua')) +
  theme_pubr() +
  theme(text=element_text(size=20), plot.title = element_text(hjust=0.5)) 
ggsave(paste0("figures_",dataset_id,"/umap_TEs_locus_clusters_SoloTE_scico_noplatelet.pdf"), device = "pdf", width=9.5, height=7)

In [ ]:
scico(30, palette = 'managua')
#>  [1] "#190C64" "#1C176B" "#202272" "#212B79" "#243580" "#263D86" "#29478B"
#>  [8] "#2C5091" "#2F5996" "#33619A" "#37699D" "#3D71A0" "#4479A1" "#4D81A2"
#> [15] "#5688A4" "#608EA2" "#6B94A1" "#77999F" "#839E9C" "#90A198" "#9BA495"
#> [22] "#A9A895" "#B7AD96" "#C7B59C" "#D7BEA6" "#E5C9B3" "#F0D4C3" "#F7DFD3"
#> [29] "#FCE9E3" "#FEF2F2"

In [ ]:
objTE_SoloTE$celltype <- celltypes$V2[match(Cells(objTE_SoloTE), celltypes$V1)]
table(objTE_SoloTE$celltype)

In [ ]:
options(repr.plot.width=9.5, repr.plot.height=7)

DimPlot(objTE_SoloTE, reduction = "umap", group.by = "celltype",
        cols=PBMCcelltypeColors,
        shuffle=T, pt.size = 1) + 
        ggtitle("Gene-derived cell types") +
  theme_pubr() +
  theme(text=element_text(size=20), plot.title = element_text(hjust=0.5)) 
ggsave(paste0("figures_",dataset_id,"/umap_TEs_locus_celltypes_SoloTE_noplatelet.pdf"), device = "pdf", width=9.5, height=7)

In [ ]:
clustersResList <- list()
identical(Cells(objGenes_SoloTE), Cells(objTE_SoloTE)) # TRUE, same cells in same order

for(res in seq(0.5, 2, by=0.1)){
    print(res)
    
    clusters_genes <- FindClusters(objGenes_SoloTE, resolution = res)
    clusters_TEs <- FindClusters(objTE_SoloTE, resolution = res)

    clustersResList[[as.character(res)]] <- cbind(clusters_genes$seurat_clusters, clusters_TEs$seurat_clusters)
    #
}


In [ ]:
library(mclust)
library(viridis)

resolutions <- seq(0.5, 2.0, by = 0.1)

ari_matrix <- matrix(data=NA, nrow=length(resolutions), ncol=length(resolutions))
colnames(ari_matrix) <- paste0("gene_r",resolutions)
rownames(ari_matrix) <- paste0("TE_r",resolutions)

for(r_gene in resolutions){
    for(r_TE in resolutions){
        ari <- adjustedRandIndex(clustersResList[[as.character(r_gene)]][,1],
                    clustersResList[[as.character(r_TE)]][,2])
        ari_matrix[paste0("TE_r",r_TE),paste0("gene_r",r_gene)] <- ari
    }
}
ari_matrix

p <- pheatmap(ari_matrix, display_numbers = T, color = mako(100, alpha = 1, begin = 0, end = 1, direction = 1)[],
         border_color = NA, number_format = "%.3f", number_color="black",
         cluster_rows = FALSE, cluster_columns = FALSE,
         cellwidth = 25, cellheight = 25)

pdf(paste0("figures_",dataset_id,"/ARI_celltypes_SoloTE_noplatelet.pdf"), width=9.5, height=9)
p
dev.off()
# ari_values <- sapply(resolutions, function(r) {
#     adjustedRandIndex(clustersResList[[as.character(r)]][,1],
#                       clustersResList[[as.character(r)]][,2])
# })

# plot(resolutions, ari_values, type = "b",
#      xlab = "Resolution", ylab = "ARI",
#      main = "Gene vs Transposon Clustering Similarity")


In [ ]:
TEclusters <- clustersResList[["1"]][,2]
table(clustersResList[["1"]][,2])

GENEclusters <- clustersResList[["1.7"]][,1]
table(clustersResList[["1.7"]][,1])

In [ ]:
options(repr.plot.width=8, repr.plot.height=7)

concordanceTable <- table(TEclusters, GENEclusters)

rownames(concordanceTable) <- paste0("TE_cluster", rownames(concordanceTable))
colnames(concordanceTable) <- paste0("GENE_cluster", colnames(concordanceTable))

# pheatmap(concordanceTable, display_numbers = T,main = "Cluster concordance - TEs VS Genes",
#         fontsize_number = 12, fontsize = 12, annotation_names_row = TRUE,
#         color = alpha(brewer.pal(9,'Purples')[1:7], 0.5),
#         cluster_rows=FALSE, cluster_cols=FALSE, 
#          border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)

In [ ]:
library(clue)
library(grid)
options(repr.plot.width=9, repr.plot.height=8)

# Suppose your matrix has more rows than columns
nr <- nrow(concordanceTable)
nc <- ncol(concordanceTable)

if(nr > nc){
  # pad with zeros to make it square
  M <- cbind(concordanceTable, matrix(0, nrow = nr, ncol = nr - nc))
} else {
  M <- rbind(concordanceTable, matrix(0, nrow = nc - nr, ncol = nc))
}

perm <- solve_LSAP(M, maximum = TRUE)

# keep only the original columns
M_reordered <- M[, perm[1:nc]]
rownames(M_reordered)


ph <- pheatmap( M_reordered[(rownames(M_reordered)!=""),!is.na(colnames(M_reordered))] , 
        display_numbers = T, color = colorRampPalette(brewer.pal(9,'Blues')[1:7])(100),#adjustcolor((colorRampPalette(carto_pal(name="Emrld")))(100),alpha.f=1),
        main = "Cluster concordance - TEs VS Genes",
        cluster_rows=FALSE, cluster_cols=FALSE, 
                fontsize_number = 12, fontsize = 12, annotation_names_row = TRUE, number_color = "black",
         border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)


# write to PDF safely

pdf(paste0("figures_", dataset_id, "/heatmap_cluster_clusters_concordance_SoloTE_noplatelet.pdf"), width=8, height=9)

grid::grid.newpage()
grid::grid.draw(ph$gtable)

dev.off()


In [ ]:
options(repr.plot.width=8, repr.plot.height=8)

concordanceTable <- table(TEclusters, GENEclusters)

rownames(concordanceTable) <- paste0("TE_cluster", rownames(concordanceTable))
colnames(concordanceTable) <- paste0("GENE_cluster", colnames(concordanceTable))


concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

# Suppose your matrix has more rows than columns
nr <- nrow(concordanceTable)
nc <- ncol(concordanceTable)
concordanceTable
if(nr > nc){
  # pad with zeros to make it square
  M <- cbind(concordanceTable, matrix(0, nrow = nr, ncol = nr - nc))
} else {
  M <- concordanceTable
}
perm <- solve_LSAP(M, maximum = TRUE)
M
perm
# keep only the original columns
M_reordered <- M[, perm]
M_reordered <- M_reordered[, colSums(M_reordered)>0]


ph <- pheatmap(M_reordered, 
        display_numbers = TRUE, color = colorRampPalette(brewer.pal(9,'Blues')[1:7])(100),#adjustcolor((colorRampPalette(carto_pal(name="Emrld")))(100),alpha.f=1),
        main = "TE cluster - cell type concordance",
        cluster_rows=FALSE, cluster_cols=FALSE, 
                fontsize_number = 12, fontsize = 12, annotation_names_row = TRUE, number_color = "black",
         border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)
dev.off()


pdf(paste0("figures_", dataset_id, "/heatmap_cluster_celltype_concordance_SoloTE_noPlatelet.pdf"), width=8, height=8)
grid::grid.newpage()
grid::grid.draw(ph$gtable)
dev.off()

In [ ]:
options(repr.plot.width=10, repr.plot.height=8)

concordanceTable <- table(TEclusters, GENEclusters)

rownames(concordanceTable) <- paste0("TE_cluster", rownames(concordanceTable))
colnames(concordanceTable) <- paste0("GENE_cluster", colnames(concordanceTable))

concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

# Suppose your matrix has more rows than columns
nr <- nrow(concordanceTable)
nc <- ncol(concordanceTable)
concordanceTable
if(nr > nc){
  # pad with zeros to make it square
  M <- cbind(concordanceTable, matrix(0, nrow = nr, ncol = nr - nc))
} else {
  M <- concordanceTable
}
perm <- solve_LSAP(M, maximum = TRUE)
M
perm
# keep only the original columns
M_reordered <- M[, perm]
M_reordered <- M_reordered[, colSums(M_reordered)>0]


ph <- pheatmap(t(M_reordered), 
        display_numbers = TRUE, color = colorRampPalette(brewer.pal(9,'Blues')[1:7])(100),#adjustcolor((colorRampPalette(carto_pal(name="Emrld")))(100),alpha.f=1),
        main = "TE cluster - cell type concordance",
        cluster_rows=TRUE, cluster_cols=TRUE, 
                fontsize_number = 12, fontsize = 12, annotation_names_row = TRUE, number_color = "black",
         border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)


pdf(paste0("figures_", dataset_id, "/heatmap_cluster_celltype_concordance_SoloTE_clustered_noPlatelet.pdf"), width=8, height=8)
grid::grid.newpage()
grid::grid.draw(ph$gtable)
dev.off()

ph <- pheatmap(t(M_reordered), 
        display_numbers = TRUE, color = colorRampPalette(brewer.pal(9,'Blues')[1:7])(100),#adjustcolor((colorRampPalette(carto_pal(name="Emrld")))(100),alpha.f=1),
        main = "TE cluster - cell type concordance",
        cluster_rows=F, cluster_cols=F,
                fontsize_number = 12, fontsize = 14, annotation_names_row = TRUE, number_color = "black", 
                angle_col=0,
         border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)

pdf(paste0("figures_", dataset_id, "/heatmap_cluster_celltype_concordance_SoloTE_horizontal_noPlatelet.pdf"), width=10, height=8)
grid::grid.newpage()
grid::grid.draw(ph$gtable)
dev.off()

rownames(M_reordered) <- paste0("TE cluster ", rownames(M_reordered))

ph <- pheatmap(t(M_reordered), 
        display_numbers = TRUE, color = colorRampPalette(brewer.pal(9,'Blues')[1:7])(100),#adjustcolor((colorRampPalette(carto_pal(name="Emrld")))(100),alpha.f=1),
        main = "TE cluster - cell type concordance",
        cluster_rows=F, cluster_cols=F,
                fontsize_number = 12, fontsize = 14, annotation_names_row = TRUE, number_color = "black", 
                angle_col=45,
         border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)

pdf(paste0("figures_", dataset_id, "/heatmap_cluster_celltype_concordance_SoloTE_horizontal_longClusterNames.pdf"), width=10, height=8)
grid::grid.newpage()
grid::grid.draw(ph$gtable)
dev.off()




In [ ]:
options(repr.plot.width=10, repr.plot.height=8)

concordanceTable <- table(TEclusters, GENEclusters)

rownames(concordanceTable) <- paste0("TE_cluster", rownames(concordanceTable))
colnames(concordanceTable) <- paste0("GENE_cluster", colnames(concordanceTable))

concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

# Suppose your matrix has more rows than columns
nr <- nrow(concordanceTable)
nc <- ncol(concordanceTable)
concordanceTable
if(nr > nc){
  # pad with zeros to make it square
  M <- cbind(concordanceTable, matrix(0, nrow = nr, ncol = nr - nc))
} else {
  M <- concordanceTable
}
perm <- solve_LSAP(M, maximum = TRUE)
M
perm
# keep only the original columns
M_reordered <- M[, perm]
M_reordered <- M_reordered[, colSums(M_reordered)>0]


ph <- pheatmap(t(M_reordered), 
        display_numbers = TRUE, color = colorRampPalette(brewer.pal(9,'Blues')[1:7])(100),#adjustcolor((colorRampPalette(carto_pal(name="Emrld")))(100),alpha.f=1),
        main = "TE cluster - cell type concordance",
        cluster_rows=TRUE, cluster_cols=TRUE, 
                fontsize_number = 12, fontsize = 12, annotation_names_row = TRUE, number_color = "black",
         border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)


pdf(paste0("figures_", dataset_id, "/heatmap_cluster_celltype_concordance_SoloTE_clustered_noplatelet.pdf"), width=8, height=8)
grid::grid.newpage()
grid::grid.draw(ph$gtable)
dev.off()

rownames(M_reordered) <- paste0("TE cluster ", rownames(M_reordered))

ph <- pheatmap(t(M_reordered), 
        display_numbers = TRUE, color = colorRampPalette(brewer.pal(9,'Blues')[1:7])(100),#adjustcolor((colorRampPalette(carto_pal(name="Emrld")))(100),alpha.f=1),
        main = "TE cluster - cell type concordance",
        cluster_rows=F, cluster_cols=F, 
                fontsize_number = 12, fontsize = 14, annotation_names_row = TRUE, number_color = "black", angle_col=45,
         border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)



pdf(paste0("figures_", dataset_id, "/heatmap_cluster_celltype_concordance_SoloTE_horizontal_noplatelet.pdf"), width=10, height=8)
grid::grid.newpage()
grid::grid.draw(ph$gtable)
dev.off()

In [ ]:
options(repr.plot.width=10, repr.plot.height=5)
# Comparison to genes

objGenes_SoloTE$celltype_num <- objGenes_SoloTE$seurat_clusters
levels(objGenes_SoloTE$celltype_num) <- 1:length(levels(objGenes_SoloTE$seurat_clusters))                          

# Add cluster information from objTE_clustered to objGenes_clustered
objGenes_SoloTE$clusters.0.1 <- GENEclusters
objGenes_SoloTE$clusters.0.2 <- TEclusters
objGenes_SoloTE$clusters.0.3 <- NULL
objGenes_SoloTE$clusters.0.4 <- NULL

# Generate the clustree plot
clustree(objGenes_SoloTE, prefix = "clusters.", node_text_size=4, node_text_angle=45) + 
  theme(text=element_text(size=15)) + 
  scale_color_manual(values=alpha(c("#f9b294","#C87D95"), 0.5)) + 
  scale_size(range = c(3,20)) +
  guides(colour = FALSE) 
ggsave(paste0("figures_",dataset_id,"/clustree_clusters.pdf"), width=10, height=7 )

objTE_SoloTE$RNAclusters <- objGenes_SoloTE$seurat_clusters

options(repr.plot.width=15, repr.plot.height=9)


# Add cluster information from objTE_clustered to objGenes_clustered
objGenes_SoloTE$clusters.0.1 <- objGenes_SoloTE$celltype
objGenes_SoloTE$clusters.0.2 <- GENEclusters
objGenes_SoloTE$clusters.0.3 <- TEclusters
objGenes_SoloTE$clusters.0.4 <- objGenes_SoloTE$celltype

concordanceTable <- table(objGenes_SoloTE$seurat_clusters, objTE_SoloTE$seurat_clusters)

pheatmap(concordanceTable, display_numbers = T, color = brewer.pal(9,'Blues')[1:6],
         border_color = NA, number_format = "%.0f", cellwidth = 25, cellheight = 25)

set.seed(123)
# Generate the clustree plot
clustree(objGenes_SoloTE, prefix = "clusters.", node_text_size=4, node_text_angle=45) + 
  theme(text=element_text(size=15)) + 
  scale_color_manual(values=alpha(c("grey90","#f9b294","#C87D95","grey90"), 0.5)) + 
  scale_size(range = c(3,20)) +
  guides(colour = FALSE) 
ggsave(paste0("figures_",dataset_id,"/clustree_clusters_allcelltypes_noplatelet.pdf"), width=10, height=7 )

objTE_SoloTE$RNAclusters <- objGenes_SoloTE$seurat_clusters


In [ ]:
PBMCcelltypeColors <- c("B"="#6D5A5D",
                        "Monocytes_CD14pos"="#4E876D",
                        "Dendritic"="#C4B3AB", #"#E78063",#"#c492a7",
                        "Monocytes_FCGR3Apos"="#81B29A",
                    # "Platelet"="#c4b3ab",
                        "T_CD4_Memory"="#84B6D6",
                        "T_CD4_Naive"="#6790B5",
                        "T_CD8"="#3D405B",
                        "NK"= "#F2CC8F")

In [ ]:
options(repr.plot.width=8, repr.plot.height=3)
concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

df <- melt(concordanceTable)
df <- df[df$value!=0,]
colnames(df) <- c("cluster","celltype","nCells")
df$cluster <- as.character(df$cluster)
df$clusterSize <- table(objTE_SoloTE$seurat_clusters)[df$cluster]
df$percentage <- as.numeric(df$nCells / df$clusterSize *100)

df$cluster <- factor(df$cluster, levels=c(0,1:length(unique(df$cluster))))


ggplot(df, aes(x=cluster, y=percentage, fill=celltype)) + 
  geom_col() + 
  xlab("TE cluster") +
  scale_fill_manual(values=PBMCcelltypeColors)+
  theme_minimal() + theme(text=element_text(size=18))
ggsave("figures_10xPBMC/clusterIdentity_barplot_SoloTE_noplatelet.pdf", width=8, height=3)

In [ ]:
options(repr.plot.width=8, repr.plot.height=3)
concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

df <- melt(concordanceTable)
df <- df[df$value!=0,]
colnames(df) <- c("cluster","celltype","nCells")
df$cluster <- as.character(df$cluster)
df$clusterSize <- table(objTE_SoloTE$seurat_clusters)[df$cluster]
df$percentage <- as.numeric(df$nCells / df$clusterSize *100)

df <- df %>%
  group_by(cluster) %>%
  mutate(main = celltype[which.max(percentage)]) %>%
  ungroup() %>%
  arrange(main, desc(percentage))
df$cluster <- factor(df$cluster, levels = unique(df$cluster))


ggplot(df, aes(x=cluster, y=percentage, fill=celltype)) + 
  geom_col() + 
  xlab("TE cluster") +
  # geom_text(data = sizes,
  #         aes(x = factor(cluster), y = 105, label = clusterSize),
  #         inherit.aes = FALSE) +
  scale_fill_manual(values=PBMCcelltypeColors)+
  theme_minimal() + theme(text=element_text(size=18))
ggsave("figures_10xPBMC/clusterIdentity_perc_barplot_SoloTE_noplatelet.pdf", width=8, height=3)

In [ ]:
options(repr.plot.width=8, repr.plot.height=3)
concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

df <- melt(concordanceTable)
df <- df[df$value!=0,]
colnames(df) <- c("cluster","celltype","nCells")
df$cluster <- as.character(df$cluster)
df$clusterSize <- table(objTE_SoloTE$seurat_clusters)[df$cluster]
df$percentage <- as.numeric(df$nCells / df$clusterSize *100)

df <- df %>%
  group_by(cluster) %>%
  mutate(main = celltype[which.max(percentage)]) %>%
  ungroup() %>%
  arrange(main, desc(percentage))
df$cluster <- factor(df$cluster, levels = unique(df$cluster))


ggplot(df, aes(x=cluster, y=nCells, fill=celltype)) + 
  geom_col() + 
  xlab("TE cluster") +
  # geom_text(data = sizes,
  #         aes(x = factor(cluster), y = 105, label = clusterSize),
  #         inherit.aes = FALSE) +
  scale_fill_manual(values=PBMCcelltypeColors)+
  theme_minimal() + theme(text=element_text(size=18))
ggsave(paste0("figures_", dataset_id,"/clusterIdentity_barplot_SoloTE_noplatelet.pdf"), width=8, height=3)

In [ ]:
options(repr.plot.width=9.5, repr.plot.height=7)

DimPlot(objTE_SoloTE, reduction = "umap", group.by = "celltype",
        cols=PBMCcelltypeColors,
        shuffle=T, pt.size = 1) + 
        ggtitle("Gene-derived cell types") +
  theme_pubr() +
  theme(text=element_text(size=20), plot.title = element_text(hjust=0.5)) 
ggsave(paste0("figures_",dataset_id,"/umap_TEs_locus_celltypes_SoloTE_noplatelet.pdf"), device = "pdf", width=9.5, height=7)

In [ ]:
sizes <- df %>% distinct(cluster, clusterSize)

ggplot(df, aes(factor(cluster), percentage, fill = celltype)) +
  geom_col() +
  geom_text(data = sizes,
            aes(x = factor(cluster), y = 105, label = clusterSize),
            inherit.aes = FALSE) +
  scale_y_continuous(limits = c(0, 110)) +
  theme_classic() +
  labs(y = "Percentage", x = "Cluster")

In [ ]:
options(repr.plot.width=8, repr.plot.height=3)
concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

df <- melt(concordanceTable)
df <- df[df$value!=0,]
colnames(df) <- c("cluster","celltype","nCells")
df$celltypeSize <- table(objTE_SoloTE$celltype)[df$celltype]
df$percentage <- as.numeric(df$nCells / df$clusterSize *100)

ggplot(df, aes(x=cluster, y=percentage, fill=celltype)) + 
  geom_col() + 
  xlab("TE cluster") +
  scale_fill_manual(values=PBMCcelltypeColors)+
  theme_minimal() + theme(text=element_text(size=18))
ggsave("figures_10xPBMC/clusterIdentity_barplot_SoloTE_noplatelet.pdf", width=8, height=3)

In [ ]:
# some way to show the number of cells in each cluster? height proportional to size? Order by size? N on top?

In [ ]:
PBMCcelltypeColors <- c("B"="#5e50a1",
                        "Monocytes_CD14pos"="#fcbe60",  
                        "Dendritic"="#3787bc",
                        "Monocytes_FCGR3Apos"="#f39b5c",
                        "T_CD4_Naive"="#aadca2" ,
                        "T_CD4_Memory"="#67c1a3",
                        "T_CD8"="#4fa4b0", 
                        "NK"= "#D4404F")

In [ ]:
≈,  "#f39b5c", "#D4404F", "#5e50a1",  "#3787bc",   "#aadca2" ,"#67c1a3","#4fa4b0",  "#FEE9AE", "#ac5d8f", "#67132C")